# STAT 764 · Meeting 17 — Boosting

**Tuesday, November 3**

On Thursday (Meeting 16) a random forest averaged 200 deep trees, each grown on its own bootstrap
sample with a few random columns at every split. Averaging trees that make different mistakes took
one tree's AUC of 0.599 to the forest's 0.651.

**The question for today.** Boosting combines trees the other way round: small trees, one after
another, each fitted to what the trees before it still get wrong. Can trees that fix each other's
mistakes beat trees that vote, and what new way of overfitting comes with them?

The page `boosting_intro.html` in this folder builds boosting by hand on thirty made-up patients
(Part 1) before this notebook does it on the readmission data. Open it in a web browser.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logit
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import export_text

# The readmission data from Meetings 9-12, with Meeting 9's split by PATIENT (split 764).
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))

# Meeting 16's sample: a random 15,000 of the training PATIENTS (about 21,000 discharges), so every
# model here can be compared with Thursday's forest. The test patients are the same as in Meetings 9-12.
rng = np.random.default_rng(16)
patients = rng.choice(enc["patient_nbr"].iloc[train].unique(), 15_000, replace=False)
fit_rows = train[np.isin(enc["patient_nbr"].iloc[train], patients)]
X_fit, y_fit = X_all.iloc[fit_rows], y_all.iloc[fit_rows].to_numpy()
groups_fit = enc["patient_nbr"].iloc[fit_rows].to_numpy()   # which patient each discharge is
X_test, y_test = X_all.iloc[test], y_all.iloc[test].to_numpy()

# To CHOOSE the number of rounds we need data the model was not fitted on, and the test patients are
# kept for one final score. So hold out a fifth of the training PATIENTS, all their stays together.
fit_i, val_i = next(GroupShuffleSplit(1, test_size=0.2, random_state=17)
                    .split(X_fit, y_fit, groups=groups_fit))


def prep():
    """Meeting 9's preprocessing: fill and scale the counts, one-hot the categories."""
    return ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
    ])


def make_pipe(model):
    """Preprocessing inside the pipeline, any model at the end."""
    return Pipeline([("prep", prep()), ("model", model)])


def calibration_slope(probs, outcome):
    """Meeting 11's slope: 1 = the spread is right; below 1 = too extreme; above 1 = too timid."""
    z = logit(np.clip(probs, 1e-6, 1 - 1e-6)).reshape(-1, 1)
    return LogisticRegression(C=1e6, max_iter=1000).fit(z, outcome).coef_[0, 0]


def scores(probs, outcome=y_test):
    """One row of a comparison table: the order (AUC) and the numbers (Brier, average, slope)."""
    return {"AUC": roc_auc_score(outcome, probs), "Brier": brier_score_loss(outcome, probs),
            "average said": probs.mean(), "calibration slope": calibration_slope(probs, outcome)}


def staged_auc(pipe, rows, outcome):
    """A fitted boosting pipeline's AUC on `rows` after round 1, 2, 3, ... : one fit, every round."""
    Z = pipe[:-1].transform(rows)                      # the preprocessing alone, learned on the fit rows
    return np.array([roc_auc_score(outcome, p[:, 1]) for p in pipe["model"].staged_predict_proba(Z)])


print(f"fit on {len(fit_rows):,} discharges from 15,000 patients · test {len(test):,} discharges"
      f" · readmitted {y_test.mean():.1%}")
print(f"to choose the rounds: fit on {len(fit_i):,} stays, hold out {len(val_i):,} stays "
      f"({len(np.unique(groups_fit[val_i])):,} patients, {y_fit[val_i].sum()} readmitted)")

## 1. What a boosted model is

Boosting starts every patient at the same prediction, then adds small trees one at a time. For a yes/no
outcome it works on the **log-odds** scale:

1. **Start** at the readmission rate among the training stays, written as log-odds.
2. **Fit a small tree** to what is still wrong for each stay: the outcome (1 or 0) minus the predicted
   chance, *y* − *p*. A tree that predicts a number is a regression tree, even though the outcome is yes/no.
3. **Add a little of it**: the learning rate times each leaf's value. Then repeat from step 2.

The fitted model stores the starting value, the learning rate and the list of trees. It does not store
the patients. Below, a model of 34 trees, each asking up to three questions in a row (8 leaves).

In [ ]:
boost34 = make_pipe(GradientBoostingClassifier(n_estimators=34, learning_rate=0.1, max_depth=3,
                                               random_state=0)).fit(X_fit, y_fit)
model = boost34["model"]

# What is stored: the start, and one regression tree per round.
start = np.log(y_fit.mean() / (1 - y_fit.mean()))          # the readmission rate as log-odds
print(f"start: readmission rate {y_fit.mean():.1%} = log-odds {start:.3f}")
print(f"trees: {model.estimators_.shape[0]}, each a {type(model.estimators_[0, 0]).__name__}"
      f" with {model.estimators_[0, 0].get_n_leaves()} leaves")

# Tree 1's questions and leaf values. The counts were scaled inside the pipeline, so its cut points are in
# standard deviations; boosting_intro.html writes the same tree in the data's own units.
names = list(boost34["prep"].get_feature_names_out())
print(export_text(model.estimators_[0, 0], feature_names=names, decimals=2))

In [ ]:
# One prediction, added up: the first test stay with at least 3 prior stays (a rule fixed in advance).
row = int(np.flatnonzero(X_test["number_inpatient"].to_numpy() >= 3)[0])
z = boost34["prep"].transform(X_test.iloc[[row]])
steps = [0.1 * tree[0].predict(z)[0] for tree in model.estimators_]   # each tree's correction
total = start + sum(steps)
print(f"test row {row}: {X_test['number_inpatient'].iloc[row]} prior stays, came back: {y_test[row]}")
print(f"start {start:.3f} + tree 1 {steps[0]:+.3f} + tree 2 {steps[1]:+.3f} + ... "
      f"= log-odds {total:.3f}")
print(f"as a chance: 1 / (1 + e^{-total:.3f}) = {1 / (1 + np.exp(-total)):.4f}"
      f"   predict_proba says {boost34.predict_proba(X_test.iloc[[row]])[0, 1]:.4f}")

## 2. The round where it turns

Every round lowers the error on the stays the trees were fitted to, and nothing in the method ever stops
it. To find the round where more trees stop helping, fit on four fifths of the training patients and
watch the held-out fifth. `staged_predict_proba` gives the predictions after every round from **one**
fit. Three learning rates, up to 1,000 rounds each (about a minute in all).

In [ ]:
STAGES = [1, 2, 3, 5, 10, 20, 30, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000]
curves = {}
for lr in (0.3, 0.1, 0.03):
    t0 = time.time()
    gb = make_pipe(GradientBoostingClassifier(n_estimators=1000, learning_rate=lr, max_depth=3,
                                              random_state=0)).fit(X_fit.iloc[fit_i], y_fit[fit_i])
    curves[lr] = {"train": staged_auc(gb, X_fit.iloc[fit_i], y_fit[fit_i]),
                  "held out": staged_auc(gb, X_fit.iloc[val_i], y_fit[val_i])}
    best = int(np.argmax(curves[lr]["held out"])) + 1
    print(f"learning rate {lr}: best round {best:>4} (rate × round {lr * best:.1f}), held-out AUC "
          f"{curves[lr]['held out'][best - 1]:.4f} · after 1,000 rounds: training "
          f"{curves[lr]['train'][-1]:.4f}, held out {curves[lr]['held out'][-1]:.4f}  ({time.time() - t0:.0f} s)")

fig, ax = plt.subplots(figsize=(7, 3.4))
for lr, style in [(0.3, ":"), (0.1, "-"), (0.03, "--")]:
    ax.plot(range(1, 1001), curves[lr]["held out"], style, label=f"held out, rate {lr}")
ax.plot(range(1, 1001), curves[0.1]["train"], color="grey", alpha=0.6, label="training, rate 0.1")
ax.set(xscale="log", xlabel="round", ylabel="AUC", ylim=(0.58, 0.86),
       title="Training AUC keeps climbing; held-out AUC turns")
ax.legend(fontsize=8)
plt.show()

The best round shrinks as the learning rate grows, and for the two smaller rates the rate times the
round stays about the same: the two settings act as one. The held-out curve is flat near its top, so the
best round is a reasonable choice, not a precise one.

## 3. One score on the test patients

Refit the chosen setting, a learning rate of 0.1 for 34 rounds, on all the training stays, and score the
test stays once, beside Thursday's forest and Meeting 9's logistic regression fitted on the same sample.

In [ ]:
boost = boost34                                      # 34 rounds, learning rate 0.1, fitted on all of X_fit
forest = make_pipe(RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=5,
                                          n_jobs=-1, random_state=0)).fit(X_fit, y_fit)
logistic = make_pipe(LogisticRegression(max_iter=2000)).fit(X_fit, y_fit)
table = pd.DataFrame({name: scores(m.predict_proba(X_test)[:, 1]) for name, m in [
    ("boosting, 34 rounds", boost), ("random forest", forest), ("logistic regression", logistic)]}).T
print(table.round(4).to_string())

## Studio

**Everything you need is in this notebook.** Sections 1–3 above explain each idea, and each question
below lists its steps. The page `boosting_intro.html` in this folder walks through the same ideas with
pictures: open it in a web browser.

| | what you practice | about |
|---|---|---|
| **Question 1** | what tree depth changes | 15 minutes |
| **Question 2** | what too many rounds do to the probabilities | 15 minutes |
| **Question 3** | Meeting 16's forest tricks, inside boosting | 15 minutes |

**All three questions count.** If you do not finish in class, finish at home: the notebook is due 24
hours after class.

**How to work.**

1. If you opened this notebook fresh, run everything above first (**Run All**). The questions use
   `make_pipe`, `scores`, `staged_auc`, `X_fit`, `y_fit`, `fit_i`, `val_i`, `X_test` and `y_test`.
2. Before each question, **write your guess** as a comment in its cell, and keep it when the answer
   disagrees. The Compare uses it.
3. Each question ends with one sentence (or a few) to answer. Write it in a comment or a markdown cell
   under its output.
4. Save and upload to Canvas under "Studio notebook — Day 17" by **Wednesday, November 4, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `NameError` for `staged_auc`, `fit_i` or `scores` | Run the notebook from the top (Run All). |
| a held-out AUC near 0.8 or 0.9 | You scored the stays the model was fitted on. Score `X_fit.iloc[val_i]` against `y_fit[val_i]`. |
| `ValueError` about the number of samples | `staged_auc(model, rows, outcome)`: the rows and the outcome must be the same stays. |
| the best round is 1,000 or 600 | That is the last round, not the best: did you take `np.argmax` of the held-out curve, plus 1? |
| question 2 runs slowly | One fit of 1,000 rounds, then `staged_predict_proba` on the test stays: about 30 seconds. Do not refit for each round. |

### 1. How deep should each tree be?

- **(a)** At a learning rate of 0.1, fit three boosted models of 600 rounds on the four fifths
  (`X_fit.iloc[fit_i]`, `y_fit[fit_i]`), with `max_depth=1` (stumps), `3` and `6`, each with
  `random_state=0`.
- **(b)** For each, `staged_auc` on the held-out fifth (`X_fit.iloc[val_i]`, `y_fit[val_i]`): the best
  round (`np.argmax(...) + 1`), the held-out AUC there, and the training AUC after 600 rounds
  (`staged_auc` on `X_fit.iloc[fit_i]`, last value).

**Answer in a sentence or two:** how does depth change the best round and how fast each overfits, and
does any depth clearly win?

### 2. What do too many rounds do to the probabilities?

- **(a)** Fit one model of 1,000 rounds (learning rate 0.1, depth 3, `random_state=0`) on **all** the
  training stays, `X_fit` and `y_fit`.
- **(b)** `staged_predict_proba` on the test stays gives its predictions after every round. Keep rounds
  34, 100, 300 and 1,000, and give each a row of `scores(...)`: AUC, Brier score, average said and
  calibration slope. One way:

  ```python
  Z_test = big[:-1].transform(X_test)
  stages = list(big["model"].staged_predict_proba(Z_test))
  pd.DataFrame({k: scores(stages[k - 1][:, 1]) for k in (34, 100, 300, 1000)}).T
  ```

This table looks at the test patients many times, so it is for **understanding** the model, not for
choosing its rounds: that was done on the held-out patients in Section 2.

**Answer in a sentence or two:** what goes wrong first as the rounds pile up, the order (AUC) or the
numbers (calibration slope), and what does a slope below 1 say about the probabilities?

### 3. Meeting 16's forest tricks, inside boosting

A random forest gives each tree random rows (a bootstrap sample) and each split a few random columns.
Boosting can borrow both: `subsample=0.5` fits each round on a random half of the rows, and
`max_features="sqrt"` lets each split consider only a random √p of the columns.

- **(a)** At a learning rate of 0.1, depth 3, 600 rounds, `random_state=0`, fit four models on the four
  fifths: plain, `subsample=0.5`, `max_features="sqrt"`, and both. Time each fit (`time.time()` before
  and after).
- **(b)** For each: the best round on the held-out fifth, the held-out AUC there, the held-out AUC after
  600 rounds, and the seconds it took. Put them in one table.

**Answer in a sentence or two:** do the forest's tricks raise boosting's best score, and what do they
change instead?

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of next class

We compare answers at the start of next class (Thursday, November 5), built from the notebooks you
upload. So leave these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: the best round and held-out AUC for each depth.
3. Question 3: your table.

## Exit ticket

> Adding more trees never makes a random forest overfit, but adding more rounds can make boosting
> overfit. In two or three sentences: why the difference?

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio notebook —
Day 17". Your two studio points need this answered: a blank exit ticket earns zero. It is not graded
for being right — it is how I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **ISLP §8.2.3** — this week's reading: boosting, Algorithm 8.2, and its three settings.

📗 **ESL §10.9–10.10** — the derivation behind "gradient" boosting; **§10.12** on shrinkage and
subsampling, the two kinds of regularization in boosting.

📗 **ISLP §8.2.4, BART** — a sum of small trees like boosting, averaged like a forest.

🎓 **Meeting 18 (Thursday)** — the bake-off: boosting, the forest, logistic regression and a model that
needs no tuning, judged six ways.